In [ ]:
# 1. CÀI ĐẶT MÔI TRƯỜNG & KHẮC PHỤC LỖI
!pip uninstall -y numpy peft accelerate transformers datasets pandas pyarrow adapters -q
!pip install -q \
  "numpy<2.0.0" \
  pandas \
  pyarrow \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

In [ ]:


import os
import time
import json
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch import nn
from google.colab import drive
from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Kết nối Drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_FFT_MFND_Final_B32"
os.makedirs(SAVE_DIR, exist_ok=True)

# Tối ưu phân bổ bộ nhớ GPU (Rất cần thiết cho FFT)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. DATASET & PREPROCESS
dataset = load_dataset("anson-huang/mirage-news")
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": torch.tensor(example["label"], dtype=torch.long)
    }

encoded_dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)
encoded_dataset.set_format("torch")

# 3. MODEL ARCHITECTURE (FULL FINE-TUNING + ALIGNMENT)
class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        # Load mô hình base, mặc định HuggingFace sẽ để requires_grad=True cho tất cả param
        self.clip = CLIPModel.from_pretrained(model_name)
        embed_dim = self.clip.config.projection_dim # 512 cho B/32

        # Phase 3: Alignment & Gating (Giữ nguyên tuyệt đối)
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        h_T, h_I = outputs.text_embeds, outputs.image_embeds

        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        logits = self.classifier(h_f)
        loss = None

        if labels is not None:
            # Classification Loss
            loss_cls = nn.CrossEntropyLoss()(logits, labels)

            # Alignment Loss (Đồng nhất với các file trước)
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)

            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

model = CLIPForMFND(model_name).to(device)

# 4. METRICS & TRAINING ARGS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16, # Batch size 16 cho bản B/32 FFT thường vẫn vừa VRAM trên T4 (16GB)
    per_device_eval_batch_size=32,
    learning_rate=1e-5, # Hạ LR xuống 1e-5 để FFT an toàn, không phá hỏng pre-trained weights
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 5. EXECUTION & HARDWARE MEASUREMENT
def get_param_stats(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

train_p, all_p = get_param_stats(model)
print(f"\n📊 Params: {all_p:,} | Trainable: {train_p:,} ({ (train_p/all_p)*100 :.4f}%)")
# Với FFT, tỷ lệ này sẽ là 100%

print("\n🚀 Training Baseline Model (B/32 + FULL FINE-TUNING)...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()
trainer.train()
train_time_min = (time.time() - start_train) / 60
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

# 6. EVALUATION & LATENCY
test_splits = sorted([s for s in dataset.keys() if "test" in s])
results = {}

for split in test_splits:
    enc_test = dataset[split].map(preprocess, remove_columns=dataset[split].column_names)
    enc_test.set_format("torch")

    start_eval = time.time()
    out = trainer.predict(enc_test)
    latency = ((time.time() - start_eval) / len(enc_test)) * 1000

    results[split] = {
        "Acc (%)": round(out.metrics["test_accuracy"]*100, 2),
        "F1 (%)": round(out.metrics["test_f1"]*100, 2),
        "Latency (ms/sample)": round(latency, 2)
    }

# Tính OOD Average
ood = [s for s in test_splits if s != "test1_nyt_mj"]
results["OOD_Average"] = {k: round(np.mean([results[s][k] for s in ood]), 2) for k in results[test_splits[0]]}

# Lưu thông số hiệu suất
results["Hardware_and_Efficiency"] = {
    "Trainable_Params": train_p,
    "Total_Params": all_p,
    "Training_Time_Min": round(train_time_min, 2),
    "Peak_VRAM_GB": round(peak_vram, 2)
}

# 7. HIỂN THỊ & LƯU
df = pd.DataFrame(results).T
print("\n" + "="*70 + "\n", df.to_markdown(), "\n" + "="*70)

with open(os.path.join(SAVE_DIR, "results_B32_FFT.json"), "w") as f:
    json.dump(results, f, indent=4)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2500 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



📊 Params: 152,853,763 | Trainable: 152,853,763 (100.0000%)

🚀 Training Baseline Model (B/32 + FULL FINE-TUNING)...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.509000,0.333941,0.896000,0.995000,0.796000,0.884444
2,0.188500,0.107096,0.976400,0.981407,0.971200,0.976277
3,0.090600,0.312191,0.937600,0.889601,0.999200,0.941221
4,0.013000,0.077765,0.986400,0.984076,0.988800,0.986433
5,0.003400,0.076768,0.987200,0.987200,0.987200,0.987200



 |                         |   Acc (%) |   F1 (%) |   Latency (ms/sample) |   Trainable_Params |   Total_Params |   Training_Time_Min |   Peak_VRAM_GB |
|:------------------------|----------:|---------:|----------------------:|-------------------:|---------------:|--------------------:|---------------:|
| test1_nyt_mj            |    100    |   100    |                 20.76 |      nan           |  nan           |              nan    |         nan    |
| test2_bbc_dalle         |     58.4  |    37.72 |                 20.63 |      nan           |  nan           |              nan    |         nan    |
| test3_cnn_dalle         |     57.6  |    34.57 |                 19.13 |      nan           |  nan           |              nan    |         nan    |
| test4_bbc_sdxl          |     74.8  |    70.14 |                 17.13 |      nan           |  nan           |              nan    |         nan    |
| test5_cnn_sdxl          |     79.4  |    76.32 |                 18.18 |      nan   

In [ ]:
from google.colab import runtime

# Sau khi đã lưu xong results.json và model checkpoint...
print("✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...")

# Lệnh này sẽ ngắt kết nối và xóa runtime hiện tại
runtime.unassign()

✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...
